# BlackPearl Morphix — Stage 2b: Does a cell's gene expression predict its tissue neighbourhood?

**Question:** given only a tumour cell's own gene-expression profile (MERFISH, ~500 genes), can the model generate the neighbourhood it actually sits in (how many T cells and host cells are nearby, how close they are), in **tissue it has never seen**?

**Pre-specified test (decided before running):**
- The model receives all (non-leaky) genes of the focal cell directly and learns which matter.
- Held-out tissue blocks (as in Stages 1–2). For each held-out cell, the model generates 16 neighbourhoods from that cell's expression; their average features are compared with the cell's **real** neighbourhood (R² and Spearman ρ per feature).
- **Controls:** (1) no expression (predict the training average), (2) **shuffled expression** (the model gets another random cell's profile), plus two simple baselines: (3) **k-nearest neighbours** in expression space and (4) **linear (ridge) regression**.
- **Leakage guard:** in imaging data a tumour cell touching a T cell can pick up some of the T cell's molecules. The main model uses only genes **not** enriched (>2×) in T cells or host cells. A second model with **all** genes shows how much those leaky genes would inflate the result.

**Caveat on interpretation:** this tests whether expression *predicts* neighbourhood. Much of the link runs the other way (the neighbourhood changes the cell's genes), so it is association, not "genes build tissue".

Runtime: **GPU (T4)**, ~30 min (two models). Needs `morphix_pfish_cells (1).csv.gz` (upload when asked); downloads the 212 MB expression table. Copy the **SUMMARY** block at the end and paste it back.

In [ ]:
import os, json, time, copy, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then restart.')
CFG = dict(n_neighbours=63, test_frac=0.2, n_blocks=6, seed=0, max_train=60000, max_test=15000,
           z_dim=32, d_model=128, n_layers=4, n_heads=4, batch_size=128, steps=12000, lr=3e-4, warmup=500,
           ema=0.999, p_drop=0.15, n_eval=1500, samples_per_cell=16, sample_steps=50, knn_k=50, leak_ratio=2.0)
RUN_ALL_GENES = True     # also train a model on all genes (shows how much leaky genes would inflate results)
rng = np.random.default_rng(CFG['seed']); torch.manual_seed(CFG['seed'])
os.makedirs('results_s2b', exist_ok=True)
S = {'config': CFG}

## 1. Data

In [ ]:
# ---------- DATA: focal tumour cells and their 63-cell neighbourhoods ----------
import os, glob
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

TYPES = ['tumour_like', 'tcell', 'host']          # neighbour classes generated by the model


def load_cells():
    cands = sorted(glob.glob('morphix_pfish_cells*.csv.gz'), key=os.path.getmtime)
    if not cands:
        from google.colab import files
        print('Upload the morphix_pfish_cells file from the SUCCESSFUL Step 2 run '
              '(it may be named "morphix_pfish_cells (1).csv.gz"):')
        up = files.upload()
        cands = [n for n in up if n.startswith('morphix_pfish_cells')]
    fn = cands[-1]
    cells = pd.read_csv(fn, index_col='cell_id')
    cells['gene'] = cells['gene'].fillna('').astype(str)
    if not (cells.cell_class == 'tumour').any():
        raise ValueError(f'{fn} has no tumour cells (old file from the failed run). Delete it and upload the newer one.')
    print('using', fn)
    return cells


def assign_classes(cells):
    """Same rule as the Step 3 signal check: 'other' cells with tumour-like counts are tumour-like,
    low-count 'other' cells (mostly mouse host) are 'host'."""
    tc = np.log10(cells['total_counts'] + 1)
    thr = np.quantile(tc[cells.cell_class == 'tumour'], 0.05)
    cls = np.where(cells.cell_class == 'tcell', 'tcell',
                   np.where((cells.cell_class == 'tumour') | (tc >= thr), 'tumour_like', 'host'))
    cells['ntype'] = pd.Categorical(cls, categories=TYPES).codes.astype(np.int64)
    return thr


def block_split_xy(xy, test_frac, n_blocks, seed):
    rng = np.random.default_rng(seed)
    lo, hi = xy.min(0), xy.max(0)
    b = np.minimum(((xy - lo) / (hi - lo + 1e-9) * n_blocks).astype(int), n_blocks - 1)
    bid = b[:, 0] * n_blocks + b[:, 1]
    counts = np.bincount(bid, minlength=n_blocks ** 2)
    test, n = [], 0
    for i in rng.permutation(np.unique(bid)):
        if n >= test_frac * len(xy):
            break
        test.append(i); n += counts[i]
    return np.isin(bid, test)


def build_patches(cells, cfg, rng):
    XY = cells[['x', 'y']].values
    tree = cKDTree(XY)
    d2, _ = tree.query(XY, k=2)
    unit = float(np.median(d2[:, 1]))
    n_nb = cfg['n_neighbours']
    is_test = block_split_xy(XY, cfg['test_frac'], cfg['n_blocks'], cfg['seed'])

    # conditions: 0 = unknown (unlabelled tumour-like cells), 1 = Control, 2.. = knockouts with enough cells
    tum = cells[(cells.cell_class == 'tumour') & ~cells.gene.isin(['multiple', 'none', ''])]
    vc = tum.gene.value_counts()
    genes = [g for g, n in vc.items() if g != 'Control' and n >= cfg['min_cells_per_gene']]
    cond_names = ['unknown', 'Control'] + sorted(genes)
    cond_id = {g: i for i, g in enumerate(cond_names)}
    cond = np.full(len(cells), -1)
    labelled = (cells.cell_class == 'tumour').values & cells.gene.isin(cond_names[1:]).values
    cond[labelled] = [cond_id[g] for g in cells.gene.values[labelled]]
    unlab = (cells.cell_class == 'other').values & (cells.ntype.values == TYPES.index('tumour_like'))
    cond[unlab] = 0

    focal = np.flatnonzero(cond >= 0)
    d, nb = tree.query(XY[focal], k=n_nb + 1)
    nb = nb[:, 1:]
    clean = (is_test[nb] == is_test[focal][:, None]).all(1)       # patch entirely inside its split
    focal, nb = focal[clean], nb[clean]
    pos = ((XY[nb] - XY[focal][:, None, :]) / unit).astype(np.float32)   # neighbours relative to the focal cell
    typ = cells.ntype.values[nb].astype(np.int64)
    c = cond[focal]
    te = is_test[focal]

    # cap the number of unlabelled patches (they teach general structure; labelled ones teach knockout effects)
    keep = np.ones(len(focal), bool)
    for split in [False, True]:
        idx = np.flatnonzero((c == 0) & (te == split))
        cap = cfg['max_unlabelled'] if not split else cfg['max_unlabelled'] // 4
        if len(idx) > cap:
            keep[rng.choice(idx, len(idx) - cap, replace=False)] = False
    data = dict(pos=pos[keep], typ=typ[keep], cond=c[keep], test=te[keep], focal=focal[keep])
    return data, cond_names, unit

In [ ]:
# ---------- DATA: expression of every cell + neighbourhoods around tumour-like focal cells ----------
import urllib.request

BASE = 'https://download.brainimagelibrary.org/0c/bd/0cbd479c521afff9/extras/tumors/processed/finaltables/'


def fetch(name, folder='pfish', tries=6):
    """Download with retries: the Brain Image Library server sometimes times out briefly."""
    import time as _t
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for k in range(tries):
        try:
            print(f'downloading {name} (attempt {k + 1}/{tries}) ...')
            req = urllib.request.Request(BASE + name, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r, open(path + '.part', 'wb') as f:
                while True:
                    chunk = r.read(1 << 22)
                    if not chunk:
                        break
                    f.write(chunk)
            os.replace(path + '.part', path)
            return path
        except Exception as e:
            wait = 15 * (k + 1)
            print(f'  failed ({e}); retrying in {wait}s')
            _t.sleep(wait)
    raise RuntimeError(f'Could not download {name} after {tries} attempts. The data server may be down; try again later.')


def load_expression(n_cells_expected, gene_cols=(3, 503)):
    """500-gene counts for every cell (rows aligned with the cells file). Step 2 found the genes in columns 3..502
    of merfishcounttable.csv; gene names come from tumorMerfish.csv. Verified here by exact row matching."""
    def read_tm(nrows):
        # read like Step 2: no forced index column (the header may be one name shorter than the data rows)
        df = pd.read_csv(fetch('tumorMerfish.csv'), nrows=nrows)
        if str(df.columns[0]).startswith('Unnamed'):
            df = df.iloc[:, 1:]
        return df
    tm_df = read_tm(400)
    names = [str(c) for c in tm_df.columns]
    full = pd.read_csv(fetch('merfishcounttable.csv'), header=None, dtype=np.float32).values
    if full.shape[0] != n_cells_expected:
        raise ValueError(f'count table has {full.shape[0]} rows, cells file has {n_cells_expected}')
    X = full[:, gene_cols[0]:gene_cols[1]]
    del full
    # verification: sample tumour rows from tumorMerfish.csv must appear exactly in X
    if len(names) != X.shape[1]:
        raise ValueError(f'tumorMerfish.csv has {len(names)} gene columns but the count table slice has {X.shape[1]}')
    tm = tm_df.apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float64)
    w = np.random.default_rng(0).integers(1, 1_000_003, size=X.shape[1]).astype(np.float64)
    sig = set(np.round(X.astype(np.float64) @ w).astype(np.int64))
    ok = np.mean([s in sig for s in np.round(tm[tm.sum(1) >= 5] @ w).astype(np.int64)])
    print(f'gene columns verified: {100 * ok:.1f}% of sample tumour rows found exactly')
    if ok < 0.9:
        raise ValueError('gene columns do not match tumorMerfish.csv; the column layout differs from Step 2')
    return X, names


def lognorm(X):
    tot = X.sum(1, keepdims=True)
    return np.log1p(X / np.maximum(tot, 1) * np.median(tot[tot > 0]))


def leaky_genes(L, cells, names, ratio=2.0, eps=0.05):
    """Pre-specified rule: drop genes whose mean (log-normalised) expression in T cells or in host cells is more than
    `ratio` x that in labelled tumour cells. These can 'leak' into a tumour cell from touching neighbours."""
    tum = (cells.cell_class == 'tumour').values
    tc = (cells.ntype == TYPES.index('tcell')).values
    host = (cells.ntype == TYPES.index('host')).values
    m_t, m_c, m_h = L[tum].mean(0), L[tc].mean(0), L[host].mean(0)
    leak = ((m_c + eps) / (m_t + eps) > ratio) | ((m_h + eps) / (m_t + eps) > ratio)
    tab = pd.DataFrame({'gene': names, 'tumour': m_t, 'tcell': m_c, 'host': m_h, 'dropped': leak})
    return leak, tab


def pca_fit(Z, k, max_rows=60000, seed=0):
    rng = np.random.default_rng(seed)
    S = Z[rng.choice(len(Z), min(max_rows, len(Z)), replace=False)]
    mu, sd = S.mean(0), S.std(0) + 1e-6
    _, _, Vt = np.linalg.svd((S - mu) / sd, full_matrices=False)
    W = Vt[:k].T
    proj = ((S - mu) / sd) @ W
    return dict(mu=mu, sd=sd, W=W, psd=proj.std(0) + 1e-6)


def std_fit(Z):
    return dict(mu=Z.mean(0), sd=Z.std(0) + 1e-3)


def std_apply(Z, P, clip=5.0):
    return np.clip((Z - P['mu']) / P['sd'], -clip, clip).astype(np.float32)


def pca_apply(Z, P):
    return (((Z - P['mu']) / P['sd']) @ P['W'] / P['psd']).astype(np.float32)


def build_expr_patches(cells, cfg, rng):
    XY = cells[['x', 'y']].values
    tree = cKDTree(XY)
    d2, _ = tree.query(XY, k=2)
    unit = float(np.median(d2[:, 1]))
    is_test = block_split_xy(XY, cfg['test_frac'], cfg['n_blocks'], cfg['seed'])
    focal_all = np.flatnonzero(cells.ntype.values == TYPES.index('tumour_like'))
    _, nb = tree.query(XY[focal_all], k=cfg['n_neighbours'] + 1)
    nb = nb[:, 1:]
    clean = (is_test[nb] == is_test[focal_all][:, None]).all(1)
    focal_all, nb = focal_all[clean], nb[clean]
    te = is_test[focal_all]
    keep = np.zeros(len(focal_all), bool)
    for split, cap in [(False, cfg['max_train']), (True, cfg['max_test'])]:
        idx = np.flatnonzero(te == split)
        keep[rng.choice(idx, min(cap, len(idx)), replace=False)] = True
    focal, nb = focal_all[keep], nb[keep]
    pos = ((XY[nb] - XY[focal][:, None, :]) / unit).astype(np.float32)
    typ = cells.ntype.values[nb].astype(np.int64)
    return dict(pos=pos, typ=typ, focal=focal, test=is_test[focal]), unit

In [ ]:
cells = load_cells()
assign_classes(cells)
X, GENES = load_expression(len(cells))
L = lognorm(X)
leak, gene_tab = leaky_genes(L, cells, GENES, CFG['leak_ratio'])
gene_tab.to_csv('results_s2b/gene_leakage_table.csv', index=False)
print(f'{leak.sum()} of {len(GENES)} genes flagged as leaky (enriched >{CFG["leak_ratio"]}x in T cells or host cells)')
print('examples dropped:', gene_tab[gene_tab.dropped].sort_values('tcell', ascending=False).gene.head(25).tolist())
for g in ['TRAC', 'CD3E', 'CD3D', 'CD8A', 'CD2', 'GZMB', 'PTPRC']:
    if g in GENES:
        print(f'  {g}: dropped={bool(leak[GENES.index(g)])}')
S['n_genes'] = len(GENES); S['n_leaky_dropped'] = int(leak.sum())
S['leaky_examples'] = gene_tab[gene_tab.dropped].sort_values('tcell', ascending=False).gene.head(40).tolist()

data, UNIT = build_expr_patches(cells, CFG, rng)
tr, te = ~data['test'], data['test']
POS_SCALE = float(data['pos'][tr].std())
print(f'focal tumour-like cells: {tr.sum()} train, {te.sum()} test | cell spacing {UNIT:.1f}')
S.update(n_train=int(tr.sum()), n_test=int(te.sum()), unit=UNIT)

## 2. Model and evaluation code

In [ ]:
# ---------- MODEL: knockout-conditioned flow matching over a cell neighbourhood ----------
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def pairwise_dist(p):
    diff = p[:, :, None, :] - p[:, None, :, :]
    return torch.sqrt((diff ** 2).sum(-1) + 1e-8)


class TimeEmbedding(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))

    def forward(self, t):
        half = self.d // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        a = 1000.0 * t[:, None] * freqs[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], dim=-1))


class DistanceBias(nn.Module):
    def __init__(self, n_heads, n_rbf=16, max_dist=4.0):
        super().__init__()
        self.register_buffer('centres', torch.linspace(0, max_dist, n_rbf))
        self.gamma = (n_rbf / max_dist) ** 2
        self.proj = nn.Linear(n_rbf, n_heads)

    def forward(self, pos):
        d = pairwise_dist(pos)
        rbf = torch.exp(-self.gamma * (d[..., None] - self.centres) ** 2)
        return self.proj(rbf).permute(0, 3, 1, 2)


class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.bias = DistanceBias(n_heads)
        self.ada = nn.Linear(d, 6 * d)
        nn.init.zeros_(self.ada.weight)
        nn.init.zeros_(self.ada.bias)

    def forward(self, h, c, pos):
        B, N, _ = h.shape
        s1, b1, g1, s2, b2, g2 = self.ada(c)[:, None].chunk(6, dim=-1)
        mask = self.bias(pos).reshape(B * self.n_heads, N, N)
        x = self.norm1(h) * (1 + s1) + b1
        h = h + g1 * self.attn(x, x, x, attn_mask=mask, need_weights=False)[0]
        x = self.norm2(h) * (1 + s2) + b2
        return h + g2 * self.mlp(x)


class ConditionalTissueFlow(nn.Module):
    """v_theta(x_t, t | knockout). The focal (knocked-out) cell is a fixed token at the origin; the 63 neighbours are
    generated around it. The knockout enters through a learned embedding added to the time embedding,
    so it modulates every block. Condition 0 = 'unknown', which doubles as the unconditional model."""
    def __init__(self, n_types, n_cond, d=128, n_layers=4, n_heads=4):
        super().__init__()
        self.dim = 2 + n_types
        self.inp = nn.Linear(self.dim, d)
        self.time = TimeEmbedding(d)
        self.cond = nn.Embedding(n_cond, d)
        nn.init.normal_(self.cond.weight, std=0.02)
        self.focal = nn.Parameter(torch.randn(1, 1, d) * 0.02)   # fixed token for the focal cell at the origin
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.out_norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, self.dim)

    def forward(self, x, t, cond):
        B = x.shape[0]
        c = self.time(t) + self.cond(cond)
        # prepend the focal cell as a fixed token at (0, 0): neighbours can 'see' it and keep a realistic distance
        h = torch.cat([self.focal.expand(B, 1, -1), self.inp(x)], dim=1) + c[:, None]
        pos = torch.cat([torch.zeros(B, 1, 2, device=x.device, dtype=x.dtype), x[..., :2]], dim=1)
        for blk in self.blocks:
            h = blk(h, c, pos)
        return self.out(self.out_norm(h))[:, 1:]               # velocities for the 63 neighbours only


def random_rotate(pos):
    """Rotate/reflect neighbourhoods about the focal cell (origin)."""
    B = pos.shape[0]
    th = torch.rand(B, device=pos.device) * 2 * math.pi
    c, s = th.cos(), th.sin()
    R = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)
    pos = pos @ R.transpose(1, 2)
    flip = (torch.rand(B, 1, device=pos.device) < 0.5).float() * 2 - 1
    return torch.cat([pos[..., :1] * flip[..., None], pos[..., 1:]], dim=-1)


def cond_flow_loss(model, pos, typ, cond, n_types, pos_scale, p_drop=0.1):
    B = pos.shape[0]
    x1 = torch.cat([pos / pos_scale, F.one_hot(typ, n_types).float()], dim=-1)
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=pos.device)
    tt = t[:, None, None]
    xt = (1 - tt) * x0 + tt * x1
    drop = torch.rand(B, device=pos.device) < p_drop          # hide the knockout -> learn the unconditional model too
    cond = torch.where(drop, torch.zeros_like(cond), cond)
    v = model(xt, t, cond)
    err = (v - (x1 - x0)) ** 2
    lp, lt = err[..., :2].mean(), err[..., 2:].mean()
    return lp + lt, {'pos': lp.item(), 'type': lt.item()}


@torch.no_grad()
def sample_cond(model, cond_id, n, n_cells, n_types, pos_scale, steps=100, batch=250, guidance=1.0, device='cpu', seed=0):
    """Generate n neighbourhoods for one condition. guidance=1: plain conditional model;
    guidance>1: classifier-free guidance (exaggerates the knockout-specific signal)."""
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    P, T = [], []
    for i in range(0, n, batch):
        m = min(batch, n - i)
        x = torch.randn(m, n_cells, 2 + n_types, device=device, generator=g)
        c = torch.full((m,), cond_id, device=device, dtype=torch.long)
        c0 = torch.zeros_like(c)
        for k in range(steps):
            t = torch.full((m,), k / steps, device=device)
            v = model(x, t, c)
            if guidance != 1.0:
                v0 = model(x, t, c0)
                v = v0 + guidance * (v - v0)
            x = x + v / steps
        P.append((x[..., :2] * pos_scale).cpu().numpy())
        T.append(x[..., 2:].argmax(-1).cpu().numpy())
    model.train()
    return np.concatenate(P), np.concatenate(T)


# ---------- MODEL: neighbourhood generation conditioned on the focal cell's expression ----------
class ExprTissueFlow(nn.Module):
    """v_theta(x_t, t | z), z = PCA summary of the focal cell's gene expression.
    A learned 'null' embedding replaces z when it is hidden (10-15% of training), giving the unconditional
    model in the same network. The focal cell is a fixed token at the origin."""
    def __init__(self, n_types, z_dim, d=128, n_layers=4, n_heads=4):
        super().__init__()
        self.dim = 2 + n_types
        self.inp = nn.Linear(self.dim, d)
        self.time = TimeEmbedding(d)
        self.zproj = nn.Sequential(nn.Linear(z_dim, d), nn.SiLU(), nn.Linear(d, d))
        self.null = nn.Parameter(torch.zeros(1, d))
        self.focal = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.out_norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, self.dim)

    def forward(self, x, t, z, keep):
        B = x.shape[0]
        k = keep[:, None].float()
        c = self.time(t) + k * self.zproj(z) + (1 - k) * self.null
        h = torch.cat([self.focal.expand(B, 1, -1), self.inp(x)], dim=1) + c[:, None]
        pos = torch.cat([torch.zeros(B, 1, 2, device=x.device, dtype=x.dtype), x[..., :2]], dim=1)
        for blk in self.blocks:
            h = blk(h, c, pos)
        return self.out(self.out_norm(h))[:, 1:]


def expr_flow_loss(model, pos, typ, z, n_types, pos_scale, p_drop=0.15, keep=None):
    B = pos.shape[0]
    x1 = torch.cat([pos / pos_scale, F.one_hot(typ, n_types).float()], dim=-1)
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=pos.device)
    tt = t[:, None, None]
    xt = (1 - tt) * x0 + tt * x1
    if keep is None:
        keep = torch.rand(B, device=pos.device) >= p_drop
    v = model(xt, t, z, keep)
    err = (v - (x1 - x0)) ** 2
    lp, lt = err[..., :2].mean(), err[..., 2:].mean()
    return lp + lt, {'pos': lp.item(), 'type': lt.item()}


@torch.no_grad()
def sample_expr(model, z, n_types, n_cells, pos_scale, steps=50, batch=1000, keep=True, device='cpu', seed=0):
    """One generated neighbourhood per row of z (repeat rows of z to get several samples per cell)."""
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    P, T = [], []
    for i in range(0, len(z), batch):
        zb = torch.as_tensor(z[i:i + batch], device=device)
        m = len(zb)
        x = torch.randn(m, n_cells, 2 + n_types, device=device, generator=g)
        kb = torch.full((m,), bool(keep), device=device)
        for k in range(steps):
            t = torch.full((m,), k / steps, device=device)
            x = x + model(x, t, zb, kb) / steps
        P.append((x[..., :2] * pos_scale).cpu().numpy())
        T.append(x[..., 2:].argmax(-1).cpu().numpy())
    model.train()
    return np.concatenate(P), np.concatenate(T)

In [ ]:
# ---------- EVALUATION: does the model reproduce knockout effects on held-out tissue? ----------
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance, pearsonr

FEATS = ['frac_tcell', 'frac_host', 'mean_dist', 'nn_dist', 'dist_to_tcell']


def patch_features(pos, typ):
    """Per-neighbourhood summaries, all relative to the focal cell at the origin (cell-spacing units)."""
    r = np.sqrt((pos ** 2).sum(-1))                      # distance of each neighbour to the focal cell
    is_t = typ == 1
    dt = np.where(is_t, r, np.inf).min(1)
    dt = np.where(np.isfinite(dt), dt, r.max(1))         # no T cell in the patch -> patch edge
    return pd.DataFrame({'frac_tcell': is_t.mean(1), 'frac_host': (typ == 2).mean(1),
                         'mean_dist': r.mean(1), 'nn_dist': r.min(1), 'dist_to_tcell': dt})


def shifts(feat_df, cond, genes_ids, control_id):
    """Mean feature of each knockout minus mean of Control (same set of cells)."""
    ctrl = feat_df[cond == control_id]
    out = {}
    for g in genes_ids:
        sel = feat_df[cond == g]
        out[g] = (sel.mean() - ctrl.mean()).values, len(sel)
    return out, ctrl


def matched_shifts(feat_df, cond, block, genes_ids, control_id):
    """Location-matched effect: for each knockout, compare its cells with Control cells from the SAME tissue
    block (as in the Step 3 signal check), then average over blocks weighted by the knockout's cells."""
    X = feat_df.values
    out = {}
    ctrl = cond == control_id
    for g in genes_ids:
        isg = cond == g
        diffs, w, var_g, var_c = [], [], [], []
        for b in np.unique(block[isg]):
            gi, ci = isg & (block == b), ctrl & (block == b)
            if ci.sum() == 0:
                continue
            diffs.append(X[gi].mean(0) - X[ci].mean(0)); w.append(gi.sum())
            var_c.append(X[ci].var(0) / ci.sum() * gi.sum() ** 2)
        if not w:
            out[g] = (np.full(X.shape[1], np.nan), 0, np.full(X.shape[1], np.nan)); continue
        w = np.array(w, float); diffs = np.array(diffs)
        n_used = int(w.sum())
        sel = X[isg & np.isin(block, [b for b in np.unique(block[isg]) if (ctrl & (block == b)).any()])]
        se = np.sqrt(sel.var(0) / n_used + np.sum(var_c, 0) / w.sum() ** 2)
        out[g] = ((w[:, None] * diffs).sum(0) / w.sum(), n_used, se)
    return out


def evaluate_matched(train_df, train_c, train_b, test_df, test_c, test_b, gen, cond_names, min_test=10):
    ctrl_id = cond_names.index('Control')
    genes = [i for i in range(2, len(cond_names)) if (test_c == i).sum() >= min_test and (train_c == i).sum() >= min_test]
    te = matched_shifts(test_df, test_c, test_b, genes, ctrl_id)
    trn = matched_shifts(train_df, train_c, train_b, genes, ctrl_id)
    sd = test_df[test_c == ctrl_id].std().values + 1e-9
    rows = []
    for g in genes:
        real, n_g, se = te[g]
        if n_g < min_test or not np.all(np.isfinite(trn[g][0])):
            continue
        model = (gen[g].mean() - gen[ctrl_id].mean()).values
        for j, f in enumerate(FEATS):
            rows.append(dict(gene=cond_names[g], feature=f, n_test_matched=n_g, real_test=real[j] / sd[j],
                             test_se=se[j] / sd[j], lookup_train=trn[g][0][j] / sd[j], model=model[j] / sd[j]))
    tab = pd.DataFrame(rows)
    y = tab.real_test.values
    summ = {'n_knockouts_evaluated': int(tab.gene.nunique()), 'n_gene_feature_pairs': len(tab),
            'noise_floor_mse': float(np.mean(tab.test_se ** 2)),
            'mse_zero': float(np.mean(y ** 2)),
            'mse_lookup': float(np.mean((tab.lookup_train - y) ** 2)),
            'mse_model': float(np.mean((tab.model - y) ** 2)),
            'r_lookup_vs_test': float(pearsonr(tab.lookup_train, y)[0]),
            'r_model_vs_test': float(pearsonr(tab.model, y)[0]),
            'r_model_vs_train': float(pearsonr(tab.model, tab.lookup_train)[0])}
    return tab, summ


def evaluate_effects(train_df, train_c, test_df, test_c, gen, cond_names, min_test=10):
    """gen: {cond_id: features DataFrame of generated neighbourhoods}. Returns long table + summary."""
    ctrl_id = cond_names.index('Control')
    genes = [i for i in range(2, len(cond_names)) if (test_c == i).sum() >= min_test and (train_c == i).sum() >= min_test]
    test_sh, test_ctrl = shifts(test_df, test_c, genes, ctrl_id)
    train_sh, _ = shifts(train_df, train_c, genes, ctrl_id)
    sd = test_ctrl.std().values + 1e-9
    rows = []
    for g in genes:
        real, n_g = test_sh[g]
        sel = test_df[test_c == g]
        se = np.sqrt(sel.var().values / n_g + test_ctrl.var().values / len(test_ctrl)) / sd
        model = (gen[g].mean() - gen[ctrl_id].mean()).values
        for j, f in enumerate(FEATS):
            rows.append(dict(gene=cond_names[g], feature=f, n_test=n_g,
                             real_test=real[j] / sd[j], test_se=se[j],
                             lookup_train=train_sh[g][0][j] / sd[j], model=model[j] / sd[j]))
    tab = pd.DataFrame(rows)
    y = tab.real_test.values
    summ = {
        'n_knockouts_evaluated': len(genes), 'n_gene_feature_pairs': len(tab),
        'noise_floor_mse (unavoidable, from test sampling error)': float(np.mean(tab.test_se ** 2)),
        'mse_zero (no-effect baseline)': float(np.mean(y ** 2)),
        'mse_lookup (train shifts as prediction)': float(np.mean((tab.lookup_train - y) ** 2)),
        'mse_model': float(np.mean((tab.model - y) ** 2)),
        'r_lookup_vs_test': float(pearsonr(tab.lookup_train, y)[0]),
        'r_model_vs_test': float(pearsonr(tab.model, y)[0]),
        'r_model_vs_train (did it learn the training effects?)': float(pearsonr(tab.model, tab.lookup_train)[0]),
    }
    return tab, summ


def realism(gen_df, test_df, train_df):
    """Per-feature W1 distance to held-out Control neighbourhoods, vs. real training Control (reference)."""
    return pd.DataFrame({'generated vs test': [wasserstein_distance(gen_df[f], test_df[f]) for f in FEATS],
                         'real train vs test (reference)': [wasserstein_distance(train_df[f], test_df[f]) for f in FEATS]},
                        index=FEATS)


# ---------- EVALUATION: per-cell prediction of the real neighbourhood on held-out tissue ----------
from scipy.stats import spearmanr


def r2(y, p):
    return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))


def knn_predict(z_tr, Y_tr, z_te, k=50):
    _, idx = cKDTree(z_tr).query(z_te, k=k)
    return Y_tr[idx].mean(1)


def ridge_predict(z_tr, Y_tr, z_te, lam=1.0):
    A = np.c_[z_tr, np.ones(len(z_tr))]
    W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Y_tr)
    return np.c_[z_te, np.ones(len(z_te))] @ W


def score_table(Y, preds):
    rows = []
    for name, P in preds.items():
        for j, f in enumerate(FEATS):
            rows.append(dict(method=name, feature=f, R2=r2(Y[:, j], P[:, j]),
                             spearman=float(spearmanr(Y[:, j], P[:, j])[0]) if np.std(P[:, j]) > 0 else 0.0))
    return pd.DataFrame(rows)

In [ ]:
Y_all = patch_features(data['pos'], data['typ']).values          # real neighbourhood features of every patch
K = len(TYPES)
eval_idx = np.flatnonzero(te)[np.random.default_rng(1).choice(te.sum(), min(CFG['n_eval'], te.sum()), replace=False)]
train_idx = np.flatnonzero(tr)
Y_te = Y_all[eval_idx]


def run_variant(gene_mask, tag):
    print(f'\n========== {tag}: {gene_mask.sum()} genes ==========')
    Lf = L[data['focal']][:, gene_mask]
    SP = std_fit(Lf[train_idx]); Z = std_apply(Lf, SP)            # model + ridge: all kept genes, standardised
    P = pca_fit(Lf[train_idx], CFG['z_dim']); Zp = pca_apply(Lf, P)  # kNN baseline: 32-d PCA (distances need low-d)
    Pt, Tt, Zt = (torch.tensor(a, device=DEVICE) for a in (data['pos'], data['typ'], Z))
    idx_tr = torch.tensor(train_idx, device=DEVICE)
    idx_te = torch.tensor(np.flatnonzero(te), device=DEVICE)
    torch.manual_seed(CFG['seed'])
    model = ExprTissueFlow(K, Z.shape[1], CFG['d_model'], CFG['n_layers'], CFG['n_heads']).to(DEVICE)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / CFG['warmup']) *
                                              0.5 * (1 + math.cos(math.pi * min(1.0, s / CFG['steps']))))
    log, t0 = [], time.time()
    for step in range(CFG['steps'] + 1):
        b = idx_tr[torch.randint(0, len(idx_tr), (CFG['batch_size'],), device=DEVICE)]
        loss, _ = expr_flow_loss(model, random_rotate(Pt[b]), Tt[b], Zt[b], K, POS_SCALE, CFG['p_drop'])
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        with torch.no_grad():
            for pe, p in zip(ema.parameters(), model.parameters()): pe.lerp_(p, 1 - CFG['ema'])
        if step % 2000 == 0:
            with torch.no_grad():
                res = {}
                for name in ['own', 'shuffled', 'none']:
                    vals = []
                    for rep in range(4):
                        torch.manual_seed(1000 + rep)
                        j = idx_te[torch.randint(0, len(idx_te), (512,), device=DEVICE)]
                        z = Zt[j] if name != 'shuffled' else Zt[j[torch.randperm(len(j), device=DEVICE)]]
                        keep = torch.full((len(j),), name != 'none', device=DEVICE)
                        torch.manual_seed(2000 + rep)
                        vals.append(expr_flow_loss(ema, Pt[j], Tt[j], z, K, POS_SCALE, keep=keep)[0].item())
                    res[name] = float(np.mean(vals))
                torch.manual_seed(CFG['seed'] + step)
            log.append(dict(step=step, train=loss.item(), **res))
            print(f'[{tag}] step {step:6d} | train {loss.item():.4f} | held-out loss: own expression {res["own"]:.4f}, '
                  f'shuffled {res["shuffled"]:.4f}, none {res["none"]:.4f} | {time.time() - t0:.0f}s')
    log = pd.DataFrame(log)

    # per-cell predictions on held-out cells
    M = CFG['samples_per_cell']
    zr = np.repeat(Z[eval_idx], M, axis=0)
    gp, gt = sample_expr(ema, zr, K, CFG['n_neighbours'], POS_SCALE, CFG['sample_steps'], device=DEVICE, seed=1)
    P_model = patch_features(gp, gt).values.reshape(len(eval_idx), M, -1).mean(1)
    zs = np.repeat(Z[np.random.default_rng(7).permutation(eval_idx)], M, axis=0)
    gp2, gt2 = sample_expr(ema, zs, K, CFG['n_neighbours'], POS_SCALE, CFG['sample_steps'], device=DEVICE, seed=2)
    P_shuf = patch_features(gp2, gt2).values.reshape(len(eval_idx), M, -1).mean(1)
    preds = {'Morphix (own expression)': P_model, 'Morphix (shuffled expression)': P_shuf,
             'no expression (train mean)': np.tile(Y_all[train_idx].mean(0), (len(eval_idx), 1)),
             f'kNN in expression space (k={CFG["knn_k"]})': knn_predict(Zp[train_idx], Y_all[train_idx], Zp[eval_idx], CFG['knn_k']),
             'ridge regression (all kept genes)': ridge_predict(Z[train_idx], Y_all[train_idx], Z[eval_idx], lam=100.0)}
    tab = score_table(Y_te, preds)
    tab['variant'] = tag
    torch.save({'model': ema.state_dict(), 'std': SP, 'pca': P, 'gene_mask': gene_mask, 'cfg': CFG, 'pos_scale': POS_SCALE},
               f'results_s2b/morphix_expr_{tag}.pt')
    return dict(log=log, tab=tab, P_model=P_model, gen=(gp, gt), Z=Z)


out = {'clean_genes': run_variant(~leak, 'clean_genes')}
if RUN_ALL_GENES:
    out['all_genes'] = run_variant(np.ones(len(GENES), bool), 'all_genes')

## 3. Results

In [ ]:
tab = pd.concat([o['tab'] for o in out.values()])
tab.to_csv('results_s2b/per_cell_scores.csv', index=False)
for v in out:
    print(f'\n=== {v}: R^2 on held-out cells (higher = better; 0 = no better than the average) ===')
    print(tab[tab.variant == v].pivot(index='method', columns='feature', values='R2').reindex(columns=FEATS).round(3).to_string())
    print(f'--- Spearman rho ---')
    print(tab[tab.variant == v].pivot(index='method', columns='feature', values='spearman').reindex(columns=FEATS).round(3).to_string())
    lg = out[v]['log'].iloc[-1]
    print(f'held-out flow loss: own {lg.own:.4f} | shuffled {lg.shuffled:.4f} | none {lg.none:.4f}')
    S[f'{v}_heldout_loss'] = {'own': float(lg.own), 'shuffled': float(lg.shuffled), 'none': float(lg.none)}
    S[f'{v}_R2'] = tab[tab.variant == v].pivot(index='method', columns='feature', values='R2').round(4).to_dict()
    S[f'{v}_spearman'] = tab[tab.variant == v].pivot(index='method', columns='feature', values='spearman').round(4).to_dict()

fig, axes = plt.subplots(1, len(out), figsize=(7 * len(out), 4.5), squeeze=False)
for a, v in zip(axes[0], out):
    p = tab[tab.variant == v].pivot(index='feature', columns='method', values='R2').reindex(FEATS)
    p.plot.bar(ax=a, width=0.8)
    a.axhline(0, c='k', lw=0.8); a.set_title(f'R² on held-out cells ({v})'); a.set_ylabel('R²')
    a.legend(fontsize=7); a.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig('results_s2b/r2_by_method.png', dpi=150); plt.show()

o = out['clean_genes']
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, j in zip(ax, [FEATS.index('frac_tcell'), FEATS.index('frac_host')]):
    a.scatter(o['P_model'][:, j], Y_te[:, j], s=6, alpha=0.4)
    a.set_xlabel(f'predicted {FEATS[j]} (from expression)'); a.set_ylabel(f'real {FEATS[j]}')
    a.set_title(f'clean genes: rho = {spearmanr(o["P_model"][:, j], Y_te[:, j])[0]:.2f}')
plt.tight_layout(); plt.savefig('results_s2b/pred_vs_real.png', dpi=150); plt.show()

## 4. Which genes go with T-cell-rich neighbourhoods? (association, not cause)

In [ ]:
gm = ~leak
G = L[data['focal'][train_idx]][:, gm]
G = (G - G.mean(0)) / (G.std(0) + 1e-6)
assoc = {}
for f in ['frac_tcell', 'frac_host']:
    y = Y_all[train_idx, FEATS.index(f)]; y = (y - y.mean()) / y.std()
    w = np.linalg.solve(G.T @ G + 1000 * np.eye(G.shape[1]), G.T @ y)
    s = pd.Series(w, index=np.array(GENES)[gm]).sort_values()
    assoc[f] = {'top_positive': s.tail(15)[::-1].round(4).to_dict(), 'top_negative': s.head(10).round(4).to_dict()}
    print(f'\n{f}: genes most associated with HIGHER values:', list(s.tail(15)[::-1].index))
    print(f'{f}: genes most associated with LOWER values:', list(s.head(10).index))
S['gene_associations_clean'] = assoc

In [ ]:
# Examples: held-out cells with the highest / lowest predicted T-cell fraction, real vs generated
o = out['clean_genes']; M = CFG['samples_per_cell']
order = np.argsort(o['P_model'][:, FEATS.index('frac_tcell')])
pick = list(order[-3:][::-1]) + list(order[:3])
COL = np.array(['#d62728', '#1f77b4', '#bbbbbb'])
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for j, i in enumerate(pick):
    gi = i * M
    for row, (pp, tt, lbl) in enumerate([(data['pos'][eval_idx[i]], data['typ'][eval_idx[i]], 'real'),
                                         (o['gen'][0][gi], o['gen'][1][gi], 'generated')]):
        a = axes[row, j]
        a.scatter(pp[:, 0], pp[:, 1], c=COL[tt], s=30, edgecolors='k', linewidths=0.3)
        a.scatter([0], [0], marker='*', s=180, c='gold', edgecolors='k')
        a.set_xlim(-6, 6); a.set_ylim(-6, 6); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
        if row == 0:
            a.set_title(('high' if j < 3 else 'low') + f' predicted T-cell fraction\n(pred {o["P_model"][i, 0]:.2f}, real {Y_te[i, 0]:.2f})', fontsize=9)
        if j == 0: a.set_ylabel(lbl)
fig.suptitle('Held-out cells: real neighbourhood (top) vs one generated from the cell\'s expression (bottom). Red tumour-like, blue T cell, grey host')
plt.tight_layout(); plt.savefig('results_s2b/examples.png', dpi=150); plt.show()

!zip -qr morphix_stage2b_results.zip results_s2b
try:
    from google.colab import files; files.download('morphix_stage2b_results.zip')
except Exception:
    pass

## How to read the result
- **Positive result:** the clean-gene model has R² clearly above 0 for T-cell and host fractions, **and** the shuffled-expression control is ≈ 0, **and** held-out loss is lower with the cell's own expression than shuffled.
- **Compare with kNN / ridge:** if a simple baseline matches the generative model, say so. The model's extra value is that it generates whole neighbourhoods, not just averages.
- **Leakage:** if all-genes ≫ clean-genes, the extra was mostly neighbour molecules leaking in; report the clean number as the result.

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(S, indent=1, default=str)[:20000])
print('===== SUMMARY END =====')